# Single-tool eval: `docker_status`

Agent loop built from scratch (no `agent.engine` import), bound to **only** the `docker_status` tool from `agent.tools.docker`. Everything else the full agent could reach for is unavailable. This measures how far the model gets on this backend alone.


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

from agent.config.settings import settings
from agent.llm.client import get_llm
from agent.tools.docker import docker_status


In [ ]:
TOOLS = [docker_status]

SYSTEM_PROMPT = """You are Oracle, a terse infrastructure-operations assistant for the Marconi lab.
You have exactly ONE tool available: `docker_status` — Return the state of a Docker container (read-only): state, status, image. Use a container name returned by list_services. Matching is separator-agnostic so 'agent_api' matches 'marconi-agent-api'.
You are read-only. Never suggest you can restart or change anything yourself;
recommend the action for a human operator instead.
If this one tool can't answer the question, say so plainly instead of guessing.
Be concise."""


In [ ]:
def _text(content) -> str:
    """Coerce an LLM message content (str or content-block list) to plain text."""
    if isinstance(content, str):
        return content.strip()
    return "".join(b.get("text", "") for b in content if isinstance(b, dict)).strip()


def _accumulate(usage: dict, ai) -> None:
    """Add one LLM turn's token counts (if the endpoint reported any) into usage."""
    reported = ai.usage_metadata or {}
    usage["input"] += reported.get("input_tokens", 0)
    usage["output"] += reported.get("output_tokens", 0)
    usage["total"] += reported.get("total_tokens", 0)


### One turn's tool calls

There is only one tool bound, so there is no name-based dispatch — the model can only ever ask for `TOOLS[0]`. The loop below still exists because a single model turn can call that one tool more than once in parallel (e.g. the same lookup with two different arguments); it is not about choosing between tools.


In [ ]:
async def run_tool_calls(ai) -> list[ToolMessage]:
    """Run every tool call the model made this turn."""
    results = []
    for call in ai.tool_calls:
        print(f"  tool call: {call['name']}({call['args']})")
        try:
            output = await TOOLS[0].ainvoke(call["args"])
        except Exception as exc:  # noqa: BLE001 - tool can raise anything
            output = f"(tool call failed: {exc})"
        results.append(ToolMessage(content=str(output), tool_call_id=call["id"]))
    return results


### Turn budget fallback

If the model is still calling tools when `settings.max_tool_steps` runs out, ask it once more, without tools, to summarize whatever it found.


In [ ]:
async def summarize_after_budget(messages: list, usage: dict) -> str:
    messages.append(
        HumanMessage(
            content="You have used the tool budget. Summarize what you found so far, "
            "state the likely cause and next steps, and note what you could not check."
        )
    )
    final = await get_llm().ainvoke(messages)
    _accumulate(usage, final)
    return _text(final.content)


### The loop itself

Ask → if the model answered with no tool calls, return it → otherwise run those tool calls and go around again, up to the turn budget. Every LLM turn's token usage is tallied in `usage` and printed as a one-line transcript once the run ends, so each call below shows what it cost.


In [ ]:
async def run_loop(question: str) -> str:
    messages: list = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=question),
    ]
    llm = get_llm().bind_tools(TOOLS)
    usage = {"input": 0, "output": 0, "total": 0}

    for turn in range(1, settings.max_tool_steps + 1):
        ai = await llm.ainvoke(messages)
        messages.append(ai)
        _accumulate(usage, ai)
        if not ai.tool_calls:
            answer = _text(ai.content)
            break
        messages.extend(await run_tool_calls(ai))
    else:
        turn = settings.max_tool_steps
        answer = await summarize_after_budget(messages, usage)

    print(
        f"[transcript] {turn} turn(s) — tokens in={usage['input']} out={usage['output']} total={usage['total']}"
    )
    return answer


## In-scope question

A question `docker_status` alone should be able to answer.


In [ ]:
answer = await run_loop("Is the container named 'agent-api' currently running?")
print(answer)


## Out-of-scope question

A question that needs evidence outside this backend. Watch whether the model admits the limit or hallucinates past it.


In [ ]:
answer = await run_loop("What do the recent logs for agent-api say?")
print(answer)
